# 🚀 ViSceT5 — PreSTU SplitOCR Pre-Training & Downstream VQA Transfer trên Kaggle

Notebook chuẩn hướng dẫn quy trình tiền huấn luyện (**PreSTU SplitOCR**) mô hình **ViSceT5** trên 2 bộ dữ liệu Scene-Text tiếng Việt (**VinText** và **EVJVQA**), trực quan hóa kết quả (Predictions + BBoxes + Attention Heatmap), và chuyển giao trọng số (Transfer Learning) sang bài toán downstream **Scene-Text VQA**.

---

### ⚙️ Cài đặt Kaggle Notebook trước khi chạy:
1. **Accelerator:** Chọn **GPU T4** (hoặc P100 / GPU T4 x 2).
2. **Internet:** Bật **Internet ON** (ở thanh cấu hình góc phải: `Settings` $\rightarrow$ `Internet: On`). *Bắt buộc để clone repo, tải packages và weights.* 
3. **Persistence:** Files lưu tại `/kaggle/working`.

---

### 📌 Các cải tiến cốt lõi đã tích hợp trong nhánh `exp/pretrain-gen-all`:
1. **Khoanh vùng Cụm Không Gian (Spatial Region Clustering for SplitOCR):**
   * Thay vì cắt chuỗi 1 chiều ngẫu nhiên khiến các từ mục tiêu bị phân tán rải rác khắp 4 góc ảnh, thuật toán chọn một anchor box và gom cụm các hộp lân cận theo khoảng cách hình học có trọng số trục $y$ để tạo thành **Target Region** (khối biển hiệu/dòng chữ cục bộ).
   * Toàn bộ từ trong vùng khoanh là **Target** (Suffix + BBoxes), các từ ngoài vùng là **Input** (Prefix Context). Điều này giúp mô hình chỉ cần tập trung chú ý vào đúng một vùng cục bộ để đọc chữ và dự đoán toạ độ, khắc phục triệt để hiện tượng underfitting do ảnh nhỏ $224 \times 224$.
2. **Phân tách từ tự nhiên (Natural Delimiter):** Các từ nối với nhau bằng khoảng trắng `" "` tự nhiên (theo Figure 4 PreSTU), chỉ duy nhất một token `</s>` ở cuối chuỗi mục tiêu. Triệt tiêu hoàn toàn hiện tượng ảo metric và chống ngắt sớm (EOS cut) khi sinh câu trả lời VQA.
3. **Cân bằng Loss ($\lambda_{\text{bbox}} = 0.3$):** BBox loss không còn lấn át Text loss. Decoder tập trung 70% vào sinh ngôn ngữ và 30% vào định vị không gian.
4. **Visual Adaptation Cho Ký Tự Tiếng Việt:** Mở băng **4 lớp cuối** của CLIP ViT (`vision_unfreeze_last_n: 4`) với Differential LR ($1\times 10^{-5}$ vs $1\times 10^{-4}$ của T5) giúp visual encoder thích ứng sâu với dấu thanh và nét chữ tiếng Việt mà không làm hỏng các bộ lọc thị giác cơ bản.
5. **Interactive Visual Inspection:** Trực quan hóa 3 khung hình song song (Ground Truth vs Prediction vs Attention Heatmap) ngay trong notebook.
6. **Seamless Downstream Transfer:** Chuyển giao trọn vẹn ViT5 Encoder, Decoder và CLIP Vision đã thích ứng sang bài toán VQA tiếng Việt.

## 1. Clone Codebase & Checkout Nhánh Pretrain
Tự động phát hiện môi trường (Kaggle hoặc Colab), đồng bộ repository từ GitHub và chuyển sang nhánh `exp/pretrain-gen-all` chứa các cải tiến mới nhất.

In [ ]:
import os
import sys

# Tự động phát hiện thư mục làm việc (Kaggle: /kaggle/working | Colab: /content)
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
REPO_DIR = os.path.join(WORK_DIR, "ViSceT5")

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Kussssssss/ViSceT5.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout exp/pretrain-gen-all
!git pull origin exp/pretrain-gen-all
!git log --oneline -3

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2. Cài Đặt Môi Trường Chuẩn (Transformers 4.45.2 Cố Định)
Gỡ các phiên bản thư viện mặc định của Kaggle và cài đặt chính xác các phiên bản tương thích từ `requirements.txt`.

In [ ]:
%%capture
!pip uninstall -y transformers peft accelerate 2>/dev/null || true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/salaniz/pycocoevalcap
!pip install -q --upgrade --no-cache-dir gdown

In [ ]:
# Kiểm tra xác nhận phiên bản môi trường
import torch
import transformers
print(f"✅ PyTorch Version: {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
print(f"✅ Transformers Version: {transformers.__version__} (Yêu cầu cố định: 4.45.2)")
if torch.cuda.is_available():
    print(f"✅ GPU Device: {torch.cuda.get_device_name(0)} (Count: {torch.cuda.device_count()})")
assert transformers.__version__.startswith("4.45"), f"Cảnh báo: Cần transformers 4.45.x để khớp kiến trúc module, hiện tại là {transformers.__version__}"

## 3. Chuẩn Bị Dữ Liệu Tiền Huấn Luyện (VinText + EVJVQA)
Tự động tải các file nén Image và OCR từ Google Drive theo cấu hình `configs/data/VinText.yaml` và `configs/data/EVJVQA.yaml`, ghép cặp ảnh-OCR và lưu cache vào `output/pretrain`.

In [ ]:
# Định vị thư mục lưu dataset CSV đồng bộ với visualize và trainer
%env OUTPUT_PATH=./output/pretrain
!python scripts/prepare_dataset.py --config configs/data/VinText.yaml,configs/data/EVJVQA.yaml

## 4. Khởi Tạo Trọng Số Mô Hình (ViT5 Base & CLIP ViT)
Khởi tạo `OpenViVQAModel`, tải các trọng số nền tảng ViT5 và CLIP-ViT, kiểm tra tính toàn vẹn số học.

In [ ]:
!python scripts/init_model.py

## 5. Chạy Huấn Luyện PreSTU SplitOCR Pre-Training

### Cấu hình tối ưu toàn diện:
* **Epochs:** 10
* **Batch size:** 4 (per device) $\times$ 4 (gradient accumulation) = Effective Batch Size 16
* **Learning rate:** $1\times 10^{-4}$ (ViT5) và $1\times 10^{-5}$ (CLIP ViT unfrozen 4 layers)
* **Loss balance:** $\lambda_{\text{bbox}} = 0.3$
* **Vision Unfreeze:** Top-4 layers (`vision_unfreeze_last_n = 4`) + post-layernorm
* **Target Split:** Spatial Region Clustering (Khoanh vùng cụm không gian)
* **Output dir:** `/kaggle/working/pretrain_output`

In [ ]:
!python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --num_train_epochs 10 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 0.0001 \
    --lambda_bbox_ce 0.3 \
    --vision_unfreeze_last_n 4 \
    --save_total_limit 1 \
    --output_dir /kaggle/working/pretrain_output \
    --logging_dir /kaggle/working/pretrain_output/logs

## 6. Trực Quan Hóa Kết Quả & Attention Heatmap (Interactive Visual Inspection)

Cell này tải checkpoint tốt nhất vừa huấn luyện và trực quan hóa 3 khung hình song song:
1. **Khung 1 (Ground-Truth):** Ảnh gốc + Hộp BBox Tiền tố (Xanh dương) + Hộp BBox Hậu tố Mục tiêu trong vùng khoanh (Xanh lá) + Chuỗi từ Suffix chuẩn.
2. **Khung 2 (Model Prediction):** Ảnh gốc + Hộp BBox Hậu tố mô hình dự đoán (Đỏ) + Chuỗi từ Suffix do ViT5 Decoder sinh ra qua Beam Search.
3. **Khung 3 (Visual Focus Attention Heatmap):** Bản đồ nhiệt chú ý không gian của Visual Search (AVF) đè lên ảnh gốc, chỉ rõ vùng mắt mô hình đang tập trung nhìn khi sinh từ vựng.

In [ ]:
import sys
import os

# Import hàm trực quan hóa PreSTU SplitOCR
from scripts.visualize_pretrain import visualize_pretrain_samples

figs = visualize_pretrain_samples(
    checkpoint="/kaggle/working/pretrain_output",
    val_csv=None,  # Tự động định vị merged_val.csv
    sample_idx=0,
    num_samples=5,
    save_dir="/kaggle/working/pretrain_output/visualizations",
    show_plot=True
)

print(f"\n✅ Đã tạo và hiển thị thành công {len(figs)} mẫu trực quan hóa!")

## 7. Chuyển Giao Sang Downstream VQA (Transfer Learning to Fine-Tuning)

Sau khi tiền huấn luyện hoàn tất, mô hình đã sẵn sàng chuyển giao tri thức sang bài toán Scene-Text VQA tiếng Việt (**ViTextVQA**).
Toàn bộ trọng số của `vit5.encoder`, `vit5.decoder`, `qa_clip.vision_model` (4 lớp thích ứng) và `visual_search` được nạp nguyên vẹn vào `training/finetune.py`.

In [ ]:
# Chạy Fine-tune downstream VQA nạp checkpoint từ bước Pretrain
# (Bỏ comment các dòng dưới đây để chạy Fine-tune)

# !python training/finetune.py configs/finetune.yaml \
#     --dataset_name "ViTextVQA" \
#     --model_name_or_path /kaggle/working/pretrain_output \
#     --num_train_epochs 5 \
#     --per_device_train_batch_size 4 \
#     --gradient_accumulation_steps 2 \
#     --learning_rate 0.00003 \
#     --output_dir /kaggle/working/finetune_output \
#     --logging_dir /kaggle/working/finetune_output/logs

## 8. Nén Checkpoint Để Tải Về & Tùy Chọn Upload HuggingFace

In [ ]:
# Nén toàn bộ checkpoint pretrain và ảnh visualizations thành file ZIP để tải về từ giao diện Kaggle
!zip -r /kaggle/working/ViSceT5_PreSTU_Pretrain.zip /kaggle/working/pretrain_output
print("✅ Đã nén thành công checkpoint tại /kaggle/working/ViSceT5_PreSTU_Pretrain.zip")

In [ ]:
# (Tùy chọn) Đăng tải trực tiếp checkpoint lên HuggingFace Hub nếu có Token
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="/kaggle/working/pretrain_output",
#     repo_id="your-username/ViSceT5-PreSTU-Pretrained",
#     repo_type="model",
#     token="your_hf_token_here"
# )